# Notebook 8 — Analisis Perbandingan Proyek FERQUEUE: Tetap Seperti Saat Ini (`v4`) vs Jika Ditambah IS-Net

Notebook ini dirancang khusus untuk menjawab pertanyaan inti pengembangan proyek **FERQUEUE**:
> **"Bagaimana performa, kecepatan, ukuran, dan kelayakan proyek FERQUEUE jika tetap seperti saat ini (`SSD MobileNet V2 v4`), dibandingkan jika ditambahkan IS-Net?"**

### Skenario Proyek yang Dibandingkan secara Setara (*Apples-to-Apples*)
| Skenario | Alur Sistem di Proyek FERQUEUE | Dampak ke Kode Proyek |
|---|---|---|
| **1. Proyek Saat Ini (`Baseline v4`)** | `Kamera RGB → SSD MobileNet V2 (v4, anotasi manual)` | Tidak ada perubahan (`receiver-v4.json`) |
| **2. Proyek + SSD Latih IS-Net (`NB7 — Runtime Ringan`)** | `Kamera RGB → SSD MobileNet V2 (dilatih dari BBox berlabel IS-Net NB7)` | Cukup `pcv.py import-model` (tanpa ubah `live_api.py`, tetap real-time) |
| **3. Proyek + IS-Net Aktif di Runtime (`IS-Net → SSD`)** *(opsional, default aktif)* | `Kamera RGB → IS-Net (ONNX segmentasi/ROI/masking) → SSD MobileNet V2 (NB7)` | Perlu menambah modul IS-Net (+~176 MB) di `live_api.py` sebelum deteksi SSD |

### Dimensi Perbandingan Lengkap
1. **Akurasi Standar COCO (`pycocotools`)**: `mAP@[0.50:0.95]`, `AP@0.50`, `AP@0.75`, `AR@100`, dan `AP@0.50` per kelas (`Mobil`, `Truk`, `Bus`).
2. **Akurasi Operasional FERQUEUE** *(conf ≥ 0.50, NMS IoU ≤ 0.45)*: Akurasi End-to-End, Precision, Recall, F1-Score, Confusion Matrix, dan **Background False Positive Rate (FPR)** pada gambar non-kendaraan.
3. **Kualitas Lokalisasi & Sensitivitas Threshold**: Distribusi IoU (TP), Confidence TP vs FP, serta Threshold Sweep (`0.25–0.80`).
4. **Dampak ke Proyek Nyata**: Breakdown **Latency End-to-End (`IS-Net ms + SSD ms`)**, **FPS Real-Time** (dibanding target kamera gate pelabuhan ≥ 10 FPS), **Total Ukuran Deployment (`MB`)**, uji statistik **McNemar**, visualisasi side-by-side & kasus menang/kalah, serta **Matriks Keputusan & Rekomendasi Otomatis**.

## Cell 1 — Setup Library, Deteksi Hardware (GPU/CPU), & Folder Output

In [ ]:
import copy
import importlib.util
import io
import json
import math
import os
import re
import shutil
import subprocess
import sys
import zipfile
from collections import Counter
from pathlib import Path
from time import perf_counter

# Toggle Skenario 3: jalankan juga simulasi proyek dengan IS-Net aktif di runtime (IS-Net -> SSD)
# - True  : Membandingkan 3 skenario lengkap (Proyek Saat Ini v4 vs SSD Latih IS-Net vs Pipeline IS-Net+SSD Runtime).
# - False : Hanya membandingkan 2 model SSD langsung (v4 vs SSD Latih IS-Net) tanpa memuat ONNX IS-Net.
RUN_RUNTIME_ISNET_SCENARIO = True

HAS_CUDA_GPU = (
    shutil.which("nvidia-smi") is not None
    and subprocess.run(["nvidia-smi"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
)

missing_pkgs = []
if importlib.util.find_spec("pycocotools") is None:
    missing_pkgs.append("pycocotools")
if RUN_RUNTIME_ISNET_SCENARIO and importlib.util.find_spec("rembg") is None:
    missing_pkgs.append("rembg[gpu]" if HAS_CUDA_GPU else "rembg[cpu]")
if missing_pkgs:
    print("Memasang dependensi:", ", ".join(missing_pkgs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, ImageOps
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display  # type: ignore[name-defined]
except NameError:
    display = print

OUT_DIR = (
    Path("/content/outputs_presentasi/nb8_perbandingan_isnet_vs_v4")
    if IN_COLAB
    else Path("./outputs_presentasi/nb8_perbandingan_isnet_vs_v4")
)
MODEL_CACHE_DIR = (
    Path("/content/ferqueue_compare_models")
    if IN_COLAB
    else Path("./outputs_presentasi/_compare_models_cache")
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_CACHE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 11.5
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

NAME_V4 = "Proyek Saat Ini (v4)"
NAME_ISNET_SSD = "Proyek + SSD IS-Net (NB7)"
NAME_ISNET_RT = "Proyek + IS-Net Runtime"

SCENARIO_COLORS = {
    NAME_V4: "#2563EB",        # Biru — Baseline proyek saat ini
    NAME_ISNET_SSD: "#10B981", # Hijau — SSD dilatih dari output IS-Net (runtime ringan)
    NAME_ISNET_RT: "#F59E0B",  # Oranye/Amber — IS-Net aktif di runtime + SSD
}
CLASS_COLORS = {
    "Mobil": (59, 130, 246),
    "Truk": (245, 158, 11),
    "Bus": (16, 185, 129),
}


def save_fig(fig, filename: str) -> Path:
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Tersimpan: {path}")
    return path


def ensure_drive_mounted() -> bool:
    if not IN_COLAB:
        return False
    if Path("/content/drive/MyDrive").is_dir():
        return True
    drive.mount("/content/drive")
    return Path("/content/drive/MyDrive").is_dir()


gpus = tf.config.list_physical_devices("GPU")
for g in gpus:
    try:
        tf.config.experimental.set_memory_growth(g, True)
    except Exception:
        pass
DEVICE_LABEL = f"GPU ({len(gpus)}x CUDA)" if gpus else "CPU"

print(f"Lingkungan : {'Google Colab' if IN_COLAB else 'Lokal'}")
print(f"TensorFlow : {tf.__version__} | Hardware: {DEVICE_LABEL}")
print(f"Skenario 3 (IS-Net Aktif di Runtime): {'AKTIF' if RUN_RUNTIME_ISNET_SCENARIO else 'NONAKTIF'}")
print(f"Folder out : {OUT_DIR.resolve()}")

## Cell 2 — Muat Model Proyek Saat Ini (`v4`), Model SSD IS-Net (`NB7`), & (Opsional) IS-Net ONNX Runtime

Secara otomatis mencari:
- **Model Saat Ini (`v4`)**: `MyDrive/ferqueue/pcv-coco-colab-v4/pcv-model-colab-v4-*.zip` atau lokal `models/mobilenet/runs/pcv-receiver-v4/exports/imported-*` / `outputs/pcv-model-colab-v4-*.zip`.
- **Model SSD IS-Net (`NB7`)**: `MyDrive/ferqueue/pcv-ssd-isnet/pcv-model-isnet-*.zip` (atau `pcv-ssd-isnet-masked/`) atau lokal `models/mobilenet/runs/pcv-receiver-isnet/exports/imported-*` / `outputs/pcv-model-isnet-*.zip`.
- Jika `RUN_RUNTIME_ISNET_SCENARIO = True`, juga memuat model **IS-Net (`isnet-general-use` ONNX)** untuk mengukur akurasi, latency tambahan (`ms`), FPS, dan ukuran file (`MB`) secara nyata bila IS-Net ikut dipasang di pipeline runtime proyek.

In [ ]:
BASELINE_MODEL_SOURCE = "auto"  # "auto" | path eksplisit ke ZIP / folder SavedModel
ISNET_MODEL_SOURCE = "auto"     # "auto" | path eksplisit ke ZIP / folder SavedModel

# Parameter post-processing produksi FERQUEUE (sesuai apps/computer-vision/configs/receiver-v4.json)
CONF_THRESHOLD = 0.50
NMS_IOU_THRESHOLD = 0.45
MAX_DETECTIONS = 10
CLASS_AGNOSTIC_NMS = True
DEFAULT_CLASSES = ["Mobil", "Truk", "Bus"]

# Parameter runtime IS-Net (Skenario 3)
ISNET_ROI_PADDING = 0.15
ISNET_MASK_BG_COLOR = (114, 114, 114)
REALTIME_TARGET_FPS = 10.0  # Batas minimum nyaman kamera gate pelabuhan (<= 100 ms/frame)


def _ckpt_sort_key(path: Path):
    m = re.search(r"ckpt-(\d+)", path.name)
    return (int(m.group(1)) if m else -1, path.stat().st_mtime if path.exists() else 0.0)


def _find_latest_match(patterns: list[str]) -> Path | None:
    hits = []
    for pat in patterns:
        p = Path(pat)
        if "*" in pat or "?" in pat:
            parent = p.parent
            if parent.exists():
                hits.extend(parent.glob(p.name))
        elif p.exists():
            hits.append(p)
    if not hits:
        return None
    hits.sort(key=_ckpt_sort_key, reverse=True)
    return hits[0]


def _dir_size_mb(directory: Path) -> float:
    return sum(f.stat().st_size for f in directory.rglob("*") if f.is_file()) / (1024.0 ** 2)


def _resolve_model(source_cfg: str, role_key: str, display_name: str, drive_patterns: list[str], local_patterns: list[str]):
    target_path = None
    if source_cfg != "auto":
        target_path = Path(source_cfg)
        if not target_path.exists():
            raise FileNotFoundError(f"{display_name}: path tidak ditemukan -> {target_path}")
    else:
        if IN_COLAB and ensure_drive_mounted():
            target_path = _find_latest_match(drive_patterns)
        if target_path is None:
            target_path = _find_latest_match(local_patterns)
        if target_path is None and IN_COLAB:
            print(f"\n{display_name} belum ditemukan otomatis di Google Drive.")
            print(f"Silakan unggah file ZIP untuk {display_name}:")
            uploaded = colab_files.upload()
            if not uploaded:
                raise RuntimeError(f"Upload dibatalkan untuk {display_name}.")
            up_name, up_bytes = next(iter(uploaded.items()))
            target_path = MODEL_CACHE_DIR / f"{role_key}_{Path(up_name).name}"
            target_path.write_bytes(up_bytes)
        if target_path is None:
            raise FileNotFoundError(
                f"{display_name} tidak ditemukan. Pola yang dicari:\n"
                f"  Drive: {drive_patterns}\n  Lokal: {local_patterns}"
            )

    meta = {
        "classes": DEFAULT_CLASSES,
        "model": "ssd_mobilenet_v2_fpnlite_320x320_coco17_tpu-8",
        "checkpoint": "unknown",
        "preprocessing": {"denoise_kernel": 0, "clahe_clip_limit": 0, "clahe_grid_size": 8, "gamma": 1.0},
    }
    zip_size_mb = None
    sharing_valid = False

    if target_path.is_file() and target_path.suffix.lower() == ".zip":
        zip_size_mb = target_path.stat().st_size / (1024.0 ** 2)
        extract_dir = MODEL_CACHE_DIR / f"extracted_{role_key}"
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(target_path) as zf:
            names = zf.namelist()
            sharing_valid = (
                "model.json" in names
                and "saved_model/saved_model.pb" in names
                and any(n.startswith("saved_model/variables/") for n in names)
            )
            zf.extractall(extract_dir)
        if (extract_dir / "model.json").is_file():
            meta.update(json.loads((extract_dir / "model.json").read_text(encoding="utf-8")))
        saved_dir = extract_dir / "saved_model"
    elif target_path.is_dir():
        saved_dir = target_path / "saved_model" if (target_path / "saved_model/saved_model.pb").is_file() else target_path
        for cand_json in (target_path / "model.json", target_path.parent.parent / "export.json"):
            if cand_json.is_file():
                meta.update(json.loads(cand_json.read_text(encoding="utf-8")))
                break
        sharing_valid = (saved_dir / "saved_model.pb").is_file() and (saved_dir / "variables").is_dir()
    else:
        raise FileNotFoundError(f"Path model tidak valid: {target_path}")

    if not (saved_dir / "saved_model.pb").is_file():
        raise FileNotFoundError(f"saved_model.pb tidak ditemukan di {saved_dir}")

    t0 = perf_counter()
    loaded = tf.saved_model.load(str(saved_dir))
    detect_fn = loaded.signatures["serving_default"]
    load_sec = perf_counter() - t0

    dummy = tf.zeros((1, 640, 640, 3), dtype=tf.uint8)
    for _ in range(5):
        _ = detect_fn(dummy)

    saved_size_mb = _dir_size_mb(saved_dir)
    return {
        "name": display_name,
        "role": role_key,
        "source_path": str(target_path),
        "saved_dir": saved_dir,
        "meta": meta,
        "detect_fn": detect_fn,
        "load_sec": round(load_sec, 2),
        "saved_size_mb": round(saved_size_mb, 2),
        "zip_size_mb": round(zip_size_mb, 2) if zip_size_mb is not None else None,
        "sharing_valid": bool(sharing_valid and re.fullmatch(r"ckpt-\d+", str(meta.get("checkpoint", "")))),
    }


baseline_drive_patterns = [
    "/content/drive/MyDrive/ferqueue/pcv-coco-colab-v4/pcv-model-colab-v4-ckpt-*.zip",
    "/content/drive/MyDrive/ferqueue/pcv-model-colab-v4-ckpt-*.zip",
    "/content/drive/MyDrive/ferqueue/pcv-coco-colab-v3/pcv-model-colab-v3-ckpt-*.zip",
]
baseline_local_patterns = [
    "models/mobilenet/runs/pcv-receiver-v4/exports/imported-ckpt-*",
    "outputs/pcv-model-colab-v4-ckpt-*.zip",
    "../../../models/mobilenet/runs/pcv-receiver-v4/exports/imported-ckpt-*",
    "../../../outputs/pcv-model-colab-v4-ckpt-*.zip",
]

isnet_drive_patterns = [
    "/content/drive/MyDrive/ferqueue/pcv-ssd-isnet/pcv-model-isnet-ckpt-*.zip",
    "/content/drive/MyDrive/ferqueue/pcv-ssd-isnet-masked/pcv-model-isnet-masked-ckpt-*.zip",
    "/content/drive/MyDrive/ferqueue/pcv-model-isnet-ckpt-*.zip",
]
isnet_local_patterns = [
    "models/mobilenet/runs/pcv-receiver-isnet/exports/imported-ckpt-*",
    "models/mobilenet/runs/pcv-receiver-isnet-masked/exports/imported-ckpt-*",
    "outputs/pcv-model-isnet-ckpt-*.zip",
    "outputs/pcv-model-isnet-masked-ckpt-*.zip",
    "../../../models/mobilenet/runs/pcv-receiver-isnet/exports/imported-ckpt-*",
    "../../../outputs/pcv-model-isnet-ckpt-*.zip",
]

model_v4 = _resolve_model(BASELINE_MODEL_SOURCE, "v4", NAME_V4, baseline_drive_patterns, baseline_local_patterns)
model_isnet = _resolve_model(ISNET_MODEL_SOURCE, "isnet", NAME_ISNET_SSD, isnet_drive_patterns, isnet_local_patterns)

CLASSES = list(model_v4["meta"]["classes"])
assert list(model_isnet["meta"]["classes"]) == CLASSES, (
    f"Urutan kelas berbeda! v4={CLASSES}, isnet={model_isnet['meta']['classes']}"
)

# Deteksi apakah model NB7 dilatih dalam mode masked atau bbox_rgb
ISNET_TRAINING_MODE = "isnet_masked_rgb" if "masked" in Path(model_isnet["source_path"]).name.lower() else "isnet_bbox_rgb"

# Muat sesi IS-Net ONNX bila Skenario 3 aktif
ISNET_SESSION = None
ISNET_ONNX_SIZE_MB = 0.0
ISNET_PROVIDER = "N/A"
if RUN_RUNTIME_ISNET_SCENARIO:
    import onnxruntime as ort
    from rembg import new_session

    if HAS_CUDA_GPU and hasattr(ort, "preload_dlls"):
        try:
            ort.preload_dlls()
        except Exception:
            pass
    avail = ort.get_available_providers()
    providers = (
        ["CUDAExecutionProvider", "CPUExecutionProvider"]
        if (HAS_CUDA_GPU and "CUDAExecutionProvider" in avail)
        else ["CPUExecutionProvider"]
    )
    print(f"\nMemuat IS-Net ONNX (isnet-general-use) untuk Skenario 3 ({providers[0]})...")
    t_is = perf_counter()
    ISNET_SESSION = new_session("isnet-general-use", providers=providers)
    ISNET_PROVIDER = ISNET_SESSION.inner_session.get_providers()[0]
    u2net_home = Path(os.environ.get("U2NET_HOME", Path.home() / ".u2net"))
    onnx_file = u2net_home / "isnet-general-use.onnx"
    ISNET_ONNX_SIZE_MB = round(onnx_file.stat().st_size / (1024.0 ** 2), 2) if onnx_file.is_file() else 176.0
    # Warmup IS-Net 2x
    _dummy_pil = Image.fromarray(np.zeros((320, 320, 3), dtype=np.uint8))
    for _ in range(2):
        _ = ISNET_SESSION.predict(_dummy_pil)
    print(f"IS-Net ONNX siap dalam {perf_counter() - t_is:.2f} dtk | Provider={ISNET_PROVIDER} | Ukuran={ISNET_ONNX_SIZE_MB:.1f} MB")

ACTIVE_SCENARIOS = [NAME_V4, NAME_ISNET_SSD] + ([NAME_ISNET_RT] if RUN_RUNTIME_ISNET_SCENARIO else [])

spec_rows = [
    {
        "Skenario Proyek": NAME_V4,
        "Checkpoint SSD": model_v4["meta"].get("checkpoint"),
        "Alur Runtime": "RGB -> SSD MobileNet V2 (v4)",
        "Ukuran Total (MB)": model_v4["saved_size_mb"],
        "Kompatibel pcv.py": "YA" if model_v4["sharing_valid"] else "PERLU CEK",
        "Ubah live_api.py?": "Tidak",
    },
    {
        "Skenario Proyek": NAME_ISNET_SSD,
        "Checkpoint SSD": model_isnet["meta"].get("checkpoint"),
        "Alur Runtime": "RGB -> SSD MobileNet V2 (Latih IS-Net NB7)",
        "Ukuran Total (MB)": model_isnet["saved_size_mb"],
        "Kompatibel pcv.py": "YA" if model_isnet["sharing_valid"] else "PERLU CEK",
        "Ubah live_api.py?": "Tidak (cukup import-model)",
    },
]
if RUN_RUNTIME_ISNET_SCENARIO:
    rt_desc = (
        "RGB -> IS-Net Masking -> SSD NB7"
        if ISNET_TRAINING_MODE == "isnet_masked_rgb"
        else "RGB -> IS-Net ROI -> SSD NB7"
    )
    spec_rows.append({
        "Skenario Proyek": NAME_ISNET_RT,
        "Checkpoint SSD": f"IS-Net ONNX + {model_isnet['meta'].get('checkpoint')}",
        "Alur Runtime": rt_desc,
        "Ukuran Total (MB)": round(model_isnet["saved_size_mb"] + ISNET_ONNX_SIZE_MB, 2),
        "Kompatibel pcv.py": "Perlu integrasi IS-Net",
        "Ubah live_api.py?": "Ya (+tahap IS-Net)",
    })

display(pd.DataFrame(spec_rows))

## Cell 3 — Muat Dataset Uji Ground Truth (`test.tfrecord` / COCO JSON) & Audit IS-Net (NB7)

Memuat dataset uji resmi FERQUEUE (`test.tfrecord` — **464 gambar**: 451 kendaraan `Mobil`/`Truk`/`Bus` + 13 gambar background non-kendaraan) agar seluruh skenario diuji pada set gambar yang 100% identik.

In [ ]:
TEST_DATA_SOURCE = "auto"  # "auto" | path ke test.tfrecord | path ke folder gambar
MAX_TEST_IMAGES = None     # None = evaluasi SEMUA gambar test; isi angka mis. 100 untuk uji cepat

ISNET_AUDIT_DATA = None
for cand_audit in [
    Path("/content/drive/MyDrive/ferqueue/pcv-ssd-isnet/audit/isnet_dataset_audit.json"),
    Path("/content/drive/MyDrive/ferqueue/pcv-ssd-isnet-masked/audit/isnet_dataset_audit.json"),
    Path("outputs/isnet_dataset_audit.json"),
    Path("../../../outputs/isnet_dataset_audit.json"),
]:
    if cand_audit.is_file():
        ISNET_AUDIT_DATA = json.loads(cand_audit.read_text(encoding="utf-8"))
        if ISNET_AUDIT_DATA.get("training_pipeline_mode"):
            ISNET_TRAINING_MODE = ISNET_AUDIT_DATA["training_pipeline_mode"]
        print(f"Audit dataset IS-Net (NB7) ditemukan: {cand_audit} (mode={ISNET_TRAINING_MODE})")
        break


def load_from_tfrecord(tfrec_path: Path, max_images: int | None = None) -> list[dict]:
    feature_spec = {
        "image/height": tf.io.FixedLenFeature([], tf.int64),
        "image/width": tf.io.FixedLenFeature([], tf.int64),
        "image/filename": tf.io.FixedLenFeature([], tf.string),
        "image/encoded": tf.io.FixedLenFeature([], tf.string),
        "image/object/bbox/xmin": tf.io.VarLenFeature(tf.float32),
        "image/object/bbox/xmax": tf.io.VarLenFeature(tf.float32),
        "image/object/bbox/ymin": tf.io.VarLenFeature(tf.float32),
        "image/object/bbox/ymax": tf.io.VarLenFeature(tf.float32),
        "image/object/class/label": tf.io.VarLenFeature(tf.int64),
    }
    samples = []
    for idx, raw in enumerate(tf.data.TFRecordDataset(str(tfrec_path))):
        if max_images is not None and idx >= max_images:
            break
        ex = tf.io.parse_single_example(raw, feature_spec)
        h = int(ex["image/height"].numpy())
        w = int(ex["image/width"].numpy())
        fname = Path(ex["image/filename"].numpy().decode("utf-8", errors="replace").replace("\\", "/")).name
        encoded = ex["image/encoded"].numpy()
        rgb = np.asarray(Image.open(io.BytesIO(encoded)).convert("RGB"))
        xmins = tf.sparse.to_dense(ex["image/object/bbox/xmin"]).numpy()
        ymins = tf.sparse.to_dense(ex["image/object/bbox/ymin"]).numpy()
        xmaxs = tf.sparse.to_dense(ex["image/object/bbox/xmax"]).numpy()
        ymaxs = tf.sparse.to_dense(ex["image/object/bbox/ymax"]).numpy()
        labels = tf.sparse.to_dense(ex["image/object/class/label"]).numpy()
        gt_boxes = []
        for x1, y1, x2, y2, cid in zip(xmins, ymins, xmaxs, ymaxs, labels):
            cid_i = int(cid)
            if 1 <= cid_i <= len(CLASSES):
                gt_boxes.append({
                    "class_id": cid_i,
                    "class": CLASSES[cid_i - 1],
                    "xyxy_px": [float(x1 * w), float(y1 * h), float(x2 * w), float(y2 * h)],
                })
        samples.append({
            "image_id": idx + 1,
            "file_name": fname,
            "width": w,
            "height": h,
            "rgb": rgb,
            "gt": gt_boxes,
            "is_background": len(gt_boxes) == 0,
        })
    return samples


def load_from_coco_json(coco_path: Path, img_dir: Path, max_images: int | None = None) -> list[dict]:
    doc = json.loads(coco_path.read_text(encoding="utf-8"))
    cat_map = {int(c["id"]): c["name"] for c in doc["categories"]}
    cls_to_id = {c: i for i, c in enumerate(CLASSES, start=1)}
    ann_by_img = {}
    for a in doc["annotations"]:
        ann_by_img.setdefault(int(a["image_id"]), []).append(a)

    samples = []
    for img_row in doc["images"]:
        if max_images is not None and len(samples) >= max_images:
            break
        p = img_dir / img_row["file_name"]
        if not p.is_file():
            continue
        with Image.open(p) as im:
            rgb = np.asarray(ImageOps.exif_transpose(im).convert("RGB"))
        h, w = rgb.shape[:2]
        gt_boxes = []
        for a in ann_by_img.get(int(img_row["id"]), []):
            cname = cat_map.get(int(a["category_id"]))
            if cname in cls_to_id:
                x, y, bw, bh = [float(v) for v in a["bbox"]]
                gt_boxes.append({
                    "class_id": cls_to_id[cname],
                    "class": cname,
                    "xyxy_px": [x, y, x + bw, y + bh],
                })
        samples.append({
            "image_id": len(samples) + 1,
            "file_name": Path(img_row["file_name"]).name,
            "width": w,
            "height": h,
            "rgb": rgb,
            "gt": gt_boxes,
            "is_background": len(gt_boxes) == 0,
        })
    return samples


def load_from_image_folder(img_dir: Path, max_images: int | None = None) -> list[dict]:
    exts = {".jpg", ".jpeg", ".png"}
    files = sorted(p for p in img_dir.rglob("*") if p.is_file() and p.suffix.lower() in exts)
    if max_images is not None:
        files = files[:max_images]
    samples = []
    for idx, p in enumerate(files, start=1):
        with Image.open(p) as im:
            rgb = np.asarray(ImageOps.exif_transpose(im).convert("RGB"))
        h, w = rgb.shape[:2]
        samples.append({
            "image_id": idx,
            "file_name": p.name,
            "width": w,
            "height": h,
            "rgb": rgb,
            "gt": [],
            "is_background": None,
        })
    return samples


HAS_GROUND_TRUTH = False
TEST_SOURCE_LABEL = ""
test_samples = []

if TEST_DATA_SOURCE != "auto":
    src_p = Path(TEST_DATA_SOURCE)
    if src_p.is_file() and src_p.suffix == ".tfrecord":
        test_samples = load_from_tfrecord(src_p, MAX_TEST_IMAGES)
        HAS_GROUND_TRUTH = True
        TEST_SOURCE_LABEL = str(src_p)
    elif src_p.is_dir():
        test_samples = load_from_image_folder(src_p, MAX_TEST_IMAGES)
        HAS_GROUND_TRUTH = False
        TEST_SOURCE_LABEL = str(src_p)
    else:
        raise FileNotFoundError(f"TEST_DATA_SOURCE tidak ditemukan: {src_p}")
else:
    tfrec_candidates = [
        Path("/content/drive/MyDrive/ferqueue/pcv-coco-colab-v4/records/test.tfrecord"),
        Path("/content/drive/MyDrive/ferqueue/pcv-coco-colab-v4/test.tfrecord"),
        Path("/content/drive/MyDrive/ferqueue/pcv-ssd-isnet/records/test.tfrecord"),
        Path("models/mobilenet/runs/pcv-coco-colab-v4/records/test.tfrecord"),
        Path("../../../models/mobilenet/runs/pcv-coco-colab-v4/records/test.tfrecord"),
    ]
    found_tfrec = next((p for p in tfrec_candidates if p.is_file()), None)
    if found_tfrec is not None:
        print(f"Memuat Ground Truth dari TFRecord: {found_tfrec} ...")
        test_samples = load_from_tfrecord(found_tfrec, MAX_TEST_IMAGES)
        HAS_GROUND_TRUTH = True
        TEST_SOURCE_LABEL = str(found_tfrec)
    else:
        coco_pairs = [
            (Path("data/processed/pcv-coco-v4/annotations/instances_test.json"), Path("data/raw/vehicles")),
            (Path("../../../data/processed/pcv-coco-v4/annotations/instances_test.json"), Path("../../../data/raw/vehicles")),
            (Path("/content/drive/MyDrive/ferqueue/pcv-coco-v4/annotations/instances_test.json"), Path("/content/drive/MyDrive/ferqueue/gambar")),
        ]
        found_coco = next(((cj, idir) for cj, idir in coco_pairs if cj.is_file() and idir.is_dir()), None)
        if found_coco is not None:
            print(f"Memuat Ground Truth dari COCO JSON: {found_coco[0]} ...")
            test_samples = load_from_coco_json(found_coco[0], found_coco[1], MAX_TEST_IMAGES)
            HAS_GROUND_TRUTH = True
            TEST_SOURCE_LABEL = str(found_coco[0])
        else:
            folder_cands = [
                Path("/content/drive/MyDrive/ferqueue/gambar"),
                Path("data/helper"),
                Path("../../../data/helper"),
            ]
            found_dir = next((d for d in folder_cands if d.is_dir()), None)
            if found_dir is not None:
                print(f"PERINGATAN: Ground Truth tidak ditemukan. Memuat gambar tanpa GT dari: {found_dir}")
                test_samples = load_from_image_folder(found_dir, MAX_TEST_IMAGES or 100)
                HAS_GROUND_TRUTH = False
                TEST_SOURCE_LABEL = str(found_dir)
            else:
                raise FileNotFoundError(
                    "Dataset uji tidak ditemukan! Pastikan test.tfrecord ada di MyDrive/ferqueue/pcv-coco-colab-v4/records/test.tfrecord."
                )

if not test_samples:
    raise RuntimeError("Dataset uji kosong.")

gt_cls_counts = Counter(b["class"] for s in test_samples for b in s["gt"])
n_bg = sum(1 for s in test_samples if s["is_background"] is True)
n_veh = sum(1 for s in test_samples if s["is_background"] is False)

print("\n=== RINGKASAN DATASET UJI ===")
print(f"Sumber          : {TEST_SOURCE_LABEL}")
print(f"Ground Truth    : {'YA (Evaluasi Penuh COCO + Operasional)' if HAS_GROUND_TRUTH else 'TIDAK (Mode Konsistensi & Latency)'}")
print(f"Total gambar    : {len(test_samples)}")
if HAS_GROUND_TRUTH:
    print(f"Gambar kendaraan: {n_veh} ({dict(gt_cls_counts)})")
    print(f"Gambar background (non-kendaraan): {n_bg}")

if ISNET_AUDIT_DATA and ISNET_AUDIT_DATA.get("isnet_vs_gt_iou_summary"):
    print("\n=== KONTEKS KUALITAS BBOX LATIH IS-NET VS GT MANUAL (DARI NB7) ===")
    print(json.dumps(ISNET_AUDIT_DATA["isnet_vs_gt_iou_summary"], indent=2))

## Cell 4 — Jalankan Simulasi Inferensi End-to-End untuk Semua Skenario Proyek

Mengukur **akurasi** dan **waktu proses nyata (`IS-Net ms` + `SSD ms`)** pada seluruh gambar uji dengan post-processing identik `ferqueue_cv/postprocessing.py` (`conf >= 0.50`, `nms_iou <= 0.45`, `max_detections = 10`, `class_agnostic_nms = True`).

In [ ]:
def preprocess_rgb(rgb: np.ndarray, cfg: dict) -> np.ndarray:
    img = rgb
    k = int(cfg.get("denoise_kernel", 0) or 0)
    if k >= 3 and k % 2 == 1:
        img = cv2.GaussianBlur(img, (k, k), 0)
    clip = float(cfg.get("clahe_clip_limit", 0) or 0)
    if clip > 0:
        grid = int(cfg.get("clahe_grid_size", 8) or 8)
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        l_ch, a_ch, b_ch = cv2.split(lab)
        l_ch = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid)).apply(l_ch)
        img = cv2.cvtColor(cv2.merge((l_ch, a_ch, b_ch)), cv2.COLOR_LAB2RGB)
    gamma = float(cfg.get("gamma", 1.0) or 1.0)
    if abs(gamma - 1.0) > 1e-3:
        lut = np.array([((i / 255.0) ** (1.0 / gamma)) * 255 for i in range(256)], dtype=np.uint8)
        img = cv2.LUT(img, lut)
    return img


def select_detections_ferqueue(
    raw_boxes_yxyx: np.ndarray,
    raw_scores: np.ndarray,
    raw_classes: np.ndarray,
    img_w: int,
    img_h: int,
    conf_th: float = CONF_THRESHOLD,
    nms_iou_th: float = NMS_IOU_THRESHOLD,
    max_det: int = MAX_DETECTIONS,
    class_agnostic: bool = CLASS_AGNOSTIC_NMS,
) -> list[dict]:
    """Identik dengan apps/computer-vision/src/ferqueue_cv/postprocessing.py."""
    candidates = [
        int(idx)
        for idx in np.argsort(-raw_scores, kind="stable")
        if raw_scores[idx] >= conf_th
        and np.isfinite(raw_scores[idx])
        and np.isfinite(raw_boxes_yxyx[idx]).all()
        and np.isfinite(raw_classes[idx])
        and raw_boxes_yxyx[idx][2] > raw_boxes_yxyx[idx][0]
        and raw_boxes_yxyx[idx][3] > raw_boxes_yxyx[idx][1]
    ]
    keep = []
    while candidates and len(keep) < max_det:
        cur = candidates.pop(0)
        keep.append(cur)
        remaining = []
        a = raw_boxes_yxyx[cur]
        for idx in candidates:
            if not class_agnostic and raw_classes[idx] != raw_classes[cur]:
                remaining.append(idx)
                continue
            b = raw_boxes_yxyx[idx]
            inter = max(0.0, min(a[2], b[2]) - max(a[0], b[0])) * max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
            union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
            if union <= 0.0 or (inter / union) <= nms_iou_th:
                remaining.append(idx)
        candidates = remaining

    out = []
    for idx in keep:
        cid = int(raw_classes[idx])
        if 1 <= cid <= len(CLASSES):
            y1, x1, y2, x2 = [float(v) for v in raw_boxes_yxyx[idx]]
            out.append({
                "class_id": cid,
                "class": CLASSES[cid - 1],
                "score": float(raw_scores[idx]),
                "xyxy_px": [x1 * img_w, y1 * img_h, x2 * img_w, y2 * img_h],
            })
    return out


def _infer_ssd_once(detect_fn, rgb_img: np.ndarray, prep_cfg: dict):
    proc = preprocess_rgb(rgb_img, prep_cfg)
    tensor = tf.convert_to_tensor(proc[None, ...], dtype=tf.uint8)
    t0 = perf_counter()
    res = detect_fn(tensor)
    count = int(res["num_detections"][0])
    boxes = res["detection_boxes"][0, :count].numpy()
    classes = res["detection_classes"][0, :count].numpy().astype(int)
    scores = res["detection_scores"][0, :count].numpy()
    ms = (perf_counter() - t0) * 1000.0
    return boxes, classes, scores, ms


def run_direct_ssd_scenario(scenario_name: str, model_bundle: dict, samples: list[dict]) -> dict:
    detect_fn = model_bundle["detect_fn"]
    prep_cfg = model_bundle["meta"].get("preprocessing", {})
    raw_outputs, prod_detections = [], []
    isnet_ms_list, ssd_ms_list, total_ms_list = [], [], []

    print(f"\n[{scenario_name}] Inferensi {model_bundle['meta'].get('checkpoint')} pada {len(samples)} gambar...")
    t_all = perf_counter()
    for i, s in enumerate(samples, start=1):
        boxes, classes, scores, ssd_ms = _infer_ssd_once(detect_fn, s["rgb"], prep_cfg)
        raw_outputs.append({"boxes_yxyx": boxes, "classes": classes, "scores": scores})
        prod_detections.append(select_detections_ferqueue(boxes, scores, classes, s["width"], s["height"]))
        isnet_ms_list.append(0.0)
        ssd_ms_list.append(ssd_ms)
        total_ms_list.append(ssd_ms)
        if i % 100 == 0 or i == len(samples):
            print(f"  {i}/{len(samples)} gambar | median={np.median(total_ms_list):.1f} ms")

    print(
        f"Selesai dalam {perf_counter() - t_all:.1f} dtk | "
        f"Mean={np.mean(total_ms_list):.2f} ms | FPS={1000.0 / np.mean(total_ms_list):.2f}"
    )
    return {
        "checkpoint": model_bundle["meta"].get("checkpoint"),
        "total_size_mb": model_bundle["saved_size_mb"],
        "raw": raw_outputs,
        "prod": prod_detections,
        "isnet_ms": np.asarray(isnet_ms_list, dtype=np.float64),
        "ssd_ms": np.asarray(ssd_ms_list, dtype=np.float64),
        "latencies_ms": np.asarray(total_ms_list, dtype=np.float64),
    }


def run_runtime_isnet_scenario(scenario_name: str, model_bundle: dict, samples: list[dict]) -> dict:
    """Simulasi nyata proyek jika IS-Net ikut dijalankan di runtime sebelum SSD MobileNet V2."""
    detect_fn = model_bundle["detect_fn"]
    prep_cfg = model_bundle["meta"].get("preprocessing", {})
    raw_outputs, prod_detections = [], []
    isnet_ms_list, ssd_ms_list, total_ms_list = [], [], []

    print(
        f"\n[{scenario_name}] Simulasi IS-Net ({ISNET_PROVIDER}) -> SSD ({model_bundle['meta'].get('checkpoint')}) "
        f"mode={ISNET_TRAINING_MODE} pada {len(samples)} gambar..."
    )
    t_all = perf_counter()
    for i, s in enumerate(samples, start=1):
        rgb = s["rgb"]
        h, w = s["height"], s["width"]

        # Tahap 1: Segmentasi IS-Net
        t_is0 = perf_counter()
        mask_pil = ISNET_SESSION.predict(Image.fromarray(rgb))[0]
        alpha = np.asarray(mask_pil, dtype=np.uint8)
        if alpha.shape[:2] != (h, w):
            alpha = cv2.resize(alpha, (w, h), interpolation=cv2.INTER_LINEAR)
        mask_bin = (alpha > 127).astype(np.uint8) * 255
        cnts, _ = cv2.findContours(mask_bin, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        isnet_ms = (perf_counter() - t_is0) * 1000.0

        # Tahap 2: Masukkan hasil IS-Net ke SSD MobileNet V2
        if ISNET_TRAINING_MODE == "isnet_masked_rgb":
            a_f = alpha[:, :, None].astype(np.float32) / 255.0
            bg = np.full_like(rgb, ISNET_MASK_BG_COLOR, dtype=np.float32)
            masked_rgb = np.clip(rgb.astype(np.float32) * a_f + bg * (1.0 - a_f), 0, 255).astype(np.uint8)
            boxes, classes, scores, ssd_ms = _infer_ssd_once(detect_fn, masked_rgb, prep_cfg)
        else:
            # Mode ROI IS-Net (padded 15%) -> SSD MobileNet V2 + fallback ke full RGB jika kosong
            roi_box = None
            if cnts:
                largest = max(cnts, key=cv2.contourArea)
                if cv2.contourArea(largest) >= 0.005 * w * h:
                    rx, ry, rw, rh = cv2.boundingRect(largest)
                    pad_x, pad_y = int(rw * ISNET_ROI_PADDING), int(rh * ISNET_ROI_PADDING)
                    x1, y1 = max(0, rx - pad_x), max(0, ry - pad_y)
                    x2, y2 = min(w, rx + rw + pad_x), min(h, ry + rh + pad_y)
                    if (x2 - x1) >= 24 and (y2 - y1) >= 24:
                        roi_box = (x1, y1, x2, y2)

            if roi_box is not None:
                x1, y1, x2, y2 = roi_box
                crop = rgb[y1:y2, x1:x2]
                c_boxes, c_classes, c_scores, ssd_ms = _infer_ssd_once(detect_fn, crop, prep_cfg)
                # Petakan koordinat ternormalisasi crop balik ke gambar penuh
                ch, cw = float(y2 - y1), float(x2 - x1)
                mapped = np.empty_like(c_boxes)
                mapped[:, 0] = (y1 + c_boxes[:, 0] * ch) / float(h)
                mapped[:, 1] = (x1 + c_boxes[:, 1] * cw) / float(w)
                mapped[:, 2] = (y1 + c_boxes[:, 2] * ch) / float(h)
                mapped[:, 3] = (x1 + c_boxes[:, 3] * cw) / float(w)
                mapped = np.clip(mapped, 0.0, 1.0)
                if c_scores.size > 0 and float(np.max(c_scores)) >= CONF_THRESHOLD:
                    boxes, classes, scores = mapped, c_classes, c_scores
                else:
                    # Fallback aman ke gambar penuh (ukur total waktu SSD juga)
                    f_boxes, f_classes, f_scores, ssd_fb_ms = _infer_ssd_once(detect_fn, rgb, prep_cfg)
                    ssd_ms += ssd_fb_ms
                    boxes, classes, scores = f_boxes, f_classes, f_scores
            else:
                boxes, classes, scores, ssd_ms = _infer_ssd_once(detect_fn, rgb, prep_cfg)

        raw_outputs.append({"boxes_yxyx": boxes, "classes": classes, "scores": scores})
        prod_detections.append(select_detections_ferqueue(boxes, scores, classes, w, h))
        isnet_ms_list.append(isnet_ms)
        ssd_ms_list.append(ssd_ms)
        total_ms_list.append(isnet_ms + ssd_ms)

        if i % 50 == 0 or i == len(samples):
            print(
                f"  {i}/{len(samples)} gambar | IS-Net={np.median(isnet_ms_list):.1f} ms + "
                f"SSD={np.median(ssd_ms_list):.1f} ms = Total {np.median(total_ms_list):.1f} ms"
            )

    print(
        f"Selesai dalam {perf_counter() - t_all:.1f} dtk | "
        f"Mean Total={np.mean(total_ms_list):.2f} ms (IS-Net {np.mean(isnet_ms_list):.1f} + SSD {np.mean(ssd_ms_list):.1f}) | "
        f"FPS={1000.0 / np.mean(total_ms_list):.2f}"
    )
    return {
        "checkpoint": f"IS-Net+{model_bundle['meta'].get('checkpoint')}",
        "total_size_mb": round(model_bundle["saved_size_mb"] + ISNET_ONNX_SIZE_MB, 2),
        "raw": raw_outputs,
        "prod": prod_detections,
        "isnet_ms": np.asarray(isnet_ms_list, dtype=np.float64),
        "ssd_ms": np.asarray(ssd_ms_list, dtype=np.float64),
        "latencies_ms": np.asarray(total_ms_list, dtype=np.float64),
    }


runs = {
    NAME_V4: run_direct_ssd_scenario(NAME_V4, model_v4, test_samples),
    NAME_ISNET_SSD: run_direct_ssd_scenario(NAME_ISNET_SSD, model_isnet, test_samples),
}
if RUN_RUNTIME_ISNET_SCENARIO:
    runs[NAME_ISNET_RT] = run_runtime_isnet_scenario(NAME_ISNET_RT, model_isnet, test_samples)

## Cell 5 — Perhitungan Metrik Akurasi Lengkap (COCO mAP, Operasional, Per Kelas, Threshold Sweep, Agreement, & Uji McNemar)

Menghitung seluruh dimensi akurasi, efisiensi, dan kesepakatan untuk membandingkan kondisi proyek saat ini (`v4`) melawan opsi penambahan IS-Net.

In [ ]:
def compute_iou_xyxy(a, b) -> float:
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    union = area_a + area_b - inter
    return float(inter / union) if union > 1e-9 else 0.0


def evaluate_coco_official(samples: list[dict], raw_list: list[dict]) -> dict:
    SUMMARY_NAMES = (
        "mAP", "AP50", "AP75",
        "mAP_small", "mAP_medium", "mAP_large",
        "AR1", "AR10", "AR100",
        "AR100_small", "AR100_medium", "AR100_large",
    )
    categories = [{"id": i, "name": c} for i, c in enumerate(CLASSES, start=1)]
    gt_doc = {"info": {}, "images": [], "annotations": [], "categories": categories}
    preds = []

    for s, raw in zip(samples, raw_list):
        img_id = s["image_id"]
        w, h = s["width"], s["height"]
        gt_doc["images"].append({"id": img_id, "file_name": s["file_name"], "width": w, "height": h})
        for g in s["gt"]:
            x1, y1, x2, y2 = g["xyxy_px"]
            bw, bh = max(0.0, x2 - x1), max(0.0, y2 - y1)
            gt_doc["annotations"].append({
                "id": len(gt_doc["annotations"]) + 1,
                "image_id": img_id,
                "category_id": g["class_id"],
                "bbox": [x1, y1, bw, bh],
                "area": bw * bh,
                "iscrowd": 0,
            })
        scale = np.array([h, w, h, w], dtype=np.float32)
        for coords, cid, score in zip(raw["boxes_yxyx"] * scale, raw["classes"], raw["scores"]):
            ymin, xmin, ymax, xmax = map(float, coords)
            if 1 <= int(cid) <= len(CLASSES):
                preds.append({
                    "image_id": img_id,
                    "category_id": int(cid),
                    "bbox": [xmin, ymin, max(0.0, xmax - xmin), max(0.0, ymax - ymin)],
                    "score": float(score),
                })

    ref = COCO()
    ref.dataset = copy.deepcopy(gt_doc)
    ref.createIndex()
    if preds:
        pred_coco = ref.loadRes(copy.deepcopy(preds))
    else:
        pred_coco = COCO()
        pred_coco.dataset = {
            "images": copy.deepcopy(ref.dataset["images"]),
            "categories": copy.deepcopy(ref.dataset["categories"]),
            "annotations": [],
        }
        pred_coco.createIndex()

    ev = COCOeval(ref, pred_coco, iouType="bbox")
    ev.evaluate()
    ev.accumulate()
    ev.summarize()

    out = {k: float(v) for k, v in zip(SUMMARY_NAMES, ev.stats)}
    precision = ev.eval["precision"]
    area_idx = ev.params.areaRngLbl.index("all")
    max_idx = list(ev.params.maxDets).index(100)
    names_by_id = {c["id"]: c["name"] for c in categories}
    out["per_class_mAP"] = {}
    out["per_class_AP50"] = {}
    for k_idx, cid in enumerate(ev.params.catIds):
        cname = names_by_id[cid]
        vals_all = precision[:, :, k_idx, area_idx, max_idx]
        valid_all = vals_all[vals_all > -1]
        out["per_class_mAP"][cname] = float(np.mean(valid_all)) if valid_all.size else 0.0
        vals_50 = precision[0, :, k_idx, area_idx, max_idx]
        valid_50 = vals_50[vals_50 > -1]
        out["per_class_AP50"][cname] = float(np.mean(valid_50)) if valid_50.size else 0.0
    return out


def evaluate_operational(samples: list[dict], dets_list: list[list[dict]], iou_th: float = 0.50) -> dict:
    cls_tp, cls_fp, cls_fn = Counter(), Counter(), Counter()
    tp_ious, tp_confs, fp_confs = [], [], []

    cm_labels = CLASSES + ["Background"]
    cm_idx = {lbl: i for i, lbl in enumerate(cm_labels)}
    cm = np.zeros((len(cm_labels), len(cm_labels)), dtype=int)

    bg_total = bg_fp_images = veh_total = veh_correct_e2e = 0
    per_image_eval = []

    for s, dets in zip(samples, dets_list):
        gt_boxes = s["gt"]
        matched_gt = set()
        ordered = sorted(dets, key=lambda d: d["score"], reverse=True)
        for d in ordered:
            best_gi, best_iou = -1, 0.0
            for gi, g in enumerate(gt_boxes):
                if gi in matched_gt or g["class"] != d["class"]:
                    continue
                iou = compute_iou_xyxy(d["xyxy_px"], g["xyxy_px"])
                if iou >= iou_th and iou > best_iou:
                    best_iou, best_gi = iou, gi
            if best_gi >= 0:
                matched_gt.add(best_gi)
                cls_tp[d["class"]] += 1
                tp_ious.append(best_iou)
                tp_confs.append(d["score"])
            else:
                cls_fp[d["class"]] += 1
                fp_confs.append(d["score"])

        for gi, g in enumerate(gt_boxes):
            if gi not in matched_gt:
                cls_fn[g["class"]] += 1

        top_det = ordered[0] if ordered else None
        if len(gt_boxes) == 0:
            bg_total += 1
            if ordered:
                bg_fp_images += 1
                cm[cm_idx["Background"], cm_idx[top_det["class"]]] += 1
                is_correct, pred_cls, pred_conf = False, top_det["class"], top_det["score"]
            else:
                cm[cm_idx["Background"], cm_idx["Background"]] += 1
                is_correct, pred_cls, pred_conf = True, "Background", 0.0
            gt_cls, top_iou = "Background", None
        else:
            veh_total += 1
            g0 = gt_boxes[0]
            gt_cls = g0["class"]
            if top_det is None:
                cm[cm_idx[gt_cls], cm_idx["Background"]] += 1
                is_correct, pred_cls, pred_conf, top_iou = False, "Background", 0.0, 0.0
            else:
                top_iou = compute_iou_xyxy(top_det["xyxy_px"], g0["xyxy_px"])
                pred_cls, pred_conf = top_det["class"], top_det["score"]
                if top_iou >= iou_th:
                    cm[cm_idx[gt_cls], cm_idx[pred_cls]] += 1
                else:
                    cm[cm_idx[gt_cls], cm_idx["Background"]] += 1
                is_correct = (pred_cls == gt_cls) and (top_iou >= iou_th)
                if is_correct:
                    veh_correct_e2e += 1

        per_image_eval.append({
            "file_name": s["file_name"],
            "gt_class": gt_cls,
            "pred_class": pred_cls,
            "pred_conf": round(float(pred_conf), 4),
            "n_dets": len(ordered),
            "top_iou": round(float(top_iou), 4) if top_iou is not None else None,
            "is_correct": bool(is_correct),
        })

    tot_tp, tot_fp, tot_fn = sum(cls_tp.values()), sum(cls_fp.values()), sum(cls_fn.values())
    prec = tot_tp / max(1, tot_tp + tot_fp)
    rec = tot_tp / max(1, tot_tp + tot_fn)
    f1 = (2 * prec * rec) / max(1e-9, prec + rec)

    per_class = {}
    for c in CLASSES:
        tp_c, fp_c, fn_c = cls_tp[c], cls_fp[c], cls_fn[c]
        p_c = tp_c / max(1, tp_c + fp_c)
        r_c = tp_c / max(1, tp_c + fn_c)
        per_class[c] = {
            "tp": tp_c, "fp": fp_c, "fn": fn_c,
            "precision": p_c, "recall": r_c,
            "f1": (2 * p_c * r_c) / max(1e-9, p_c + r_c),
        }

    e2e_correct = veh_correct_e2e + (bg_total - bg_fp_images)
    bg_fpr = (bg_fp_images / bg_total) if bg_total > 0 else 0.0
    return {
        "tp": tot_tp, "fp": tot_fp, "fn": tot_fn,
        "precision": prec, "recall": rec, "f1": f1,
        "e2e_accuracy": e2e_correct / max(1, len(samples)),
        "vehicle_accuracy": veh_correct_e2e / max(1, veh_total),
        "bg_total": bg_total, "bg_fp_images": bg_fp_images,
        "bg_fpr": bg_fpr, "bg_specificity": 1.0 - bg_fpr,
        "mean_iou": float(np.mean(tp_ious)) if tp_ious else 0.0,
        "median_iou": float(np.median(tp_ious)) if tp_ious else 0.0,
        "pct_iou_ge_75": float(np.mean(np.asarray(tp_ious) >= 0.75)) if tp_ious else 0.0,
        "tp_ious": np.asarray(tp_ious, dtype=np.float64),
        "tp_confs": np.asarray(tp_confs, dtype=np.float64),
        "fp_confs": np.asarray(fp_confs, dtype=np.float64),
        "mean_tp_conf": float(np.mean(tp_confs)) if tp_confs else 0.0,
        "mean_fp_conf": float(np.mean(fp_confs)) if fp_confs else 0.0,
        "per_class": per_class,
        "cm": cm, "cm_labels": cm_labels,
        "per_image": per_image_eval,
    }


def exact_mcnemar_p(b: int, c: int) -> float:
    """Two-sided exact binomial test untuk statistik McNemar."""
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2.0 * sum(math.comb(n, i) for i in range(k + 1)) / (2 ** n))


metrics_by_scenario = {}
sweep_rows = []
THRESHOLDS = [round(0.25 + 0.05 * i, 2) for i in range(12)]

for sc_name in ACTIVE_SCENARIOS:
    run = runs[sc_name]
    lat = run["latencies_ms"]
    entry = {
        "checkpoint": run["checkpoint"],
        "total_size_mb": run["total_size_mb"],
        "isnet_mean_ms": float(np.mean(run["isnet_ms"])),
        "ssd_mean_ms": float(np.mean(run["ssd_ms"])),
        "mean_ms": float(np.mean(lat)),
        "median_ms": float(np.median(lat)),
        "p95_ms": float(np.percentile(lat, 95)),
        "fps": float(1000.0 / np.mean(lat)),
        "realtime_ok": bool((1000.0 / np.mean(lat)) >= REALTIME_TARGET_FPS),
        "detection_rate": float(np.mean([len(d) > 0 for d in run["prod"]])),
        "mean_top_conf": float(np.mean([d[0]["score"] for d in run["prod"] if d])) if any(run["prod"]) else 0.0,
    }
    if HAS_GROUND_TRUTH:
        print(f"\n--- Evaluasi COCO Resmi: {sc_name} ---")
        entry["coco"] = evaluate_coco_official(test_samples, run["raw"])
        entry["op"] = evaluate_operational(test_samples, run["prod"], iou_th=0.50)

        best_f1, best_th = -1.0, CONF_THRESHOLD
        for th in THRESHOLDS:
            dets_th = [
                select_detections_ferqueue(r["boxes_yxyx"], r["scores"], r["classes"], s["width"], s["height"], conf_th=th)
                for s, r in zip(test_samples, run["raw"])
            ]
            op_th = evaluate_operational(test_samples, dets_th, iou_th=0.50)
            sweep_rows.append({
                "Skenario": sc_name,
                "Threshold": th,
                "Precision_pct": round(100.0 * op_th["precision"], 2),
                "Recall_pct": round(100.0 * op_th["recall"], 2),
                "F1_pct": round(100.0 * op_th["f1"], 2),
                "E2E_Acc_pct": round(100.0 * op_th["e2e_accuracy"], 2),
                "BG_FPR_pct": round(100.0 * op_th["bg_fpr"], 2),
            })
            if op_th["f1"] > best_f1:
                best_f1, best_th = op_th["f1"], th
        entry["best_f1_sweep"] = best_f1
        entry["best_th_sweep"] = best_th
    metrics_by_scenario[sc_name] = entry

sweep_df = pd.DataFrame(sweep_rows)

# Detail per gambar & Agreement (Proyek Saat Ini v4 vs Proyek + SSD IS-Net NB7)
detail_rows = []
both_ok = v4_only_ok = isnet_only_ok = both_wrong = 0
inter_model_ious = []
class_agreements = 0

for idx, s in enumerate(test_samples):
    d_v4 = runs[NAME_V4]["prod"][idx]
    d_is = runs[NAME_ISNET_SSD]["prod"][idx]
    top_v4 = d_v4[0] if d_v4 else None
    top_is = d_is[0] if d_is else None

    cls_v4 = top_v4["class"] if top_v4 else "Background"
    cls_is = top_is["class"] if top_is else "Background"
    if cls_v4 == cls_is:
        class_agreements += 1
    pair_iou = compute_iou_xyxy(top_v4["xyxy_px"], top_is["xyxy_px"]) if (top_v4 and top_is) else None
    if pair_iou is not None:
        inter_model_ious.append(pair_iou)

    row = {
        "image_id": s["image_id"],
        "file_name": s["file_name"],
        "gt_class": s["gt"][0]["class"] if s["gt"] else ("Background" if HAS_GROUND_TRUTH else "Unknown"),
        "v4_class": cls_v4,
        "v4_conf": round(top_v4["score"], 4) if top_v4 else 0.0,
        "isnet_ssd_class": cls_is,
        "isnet_ssd_conf": round(top_is["score"], 4) if top_is else 0.0,
        "v4_ms": round(float(runs[NAME_V4]["latencies_ms"][idx]), 2),
        "isnet_ssd_ms": round(float(runs[NAME_ISNET_SSD]["latencies_ms"][idx]), 2),
        "iou_v4_vs_isnet_ssd": round(pair_iou, 4) if pair_iou is not None else None,
        "class_agree": cls_v4 == cls_is,
    }
    if RUN_RUNTIME_ISNET_SCENARIO:
        d_rt = runs[NAME_ISNET_RT]["prod"][idx]
        top_rt = d_rt[0] if d_rt else None
        row["isnet_rt_class"] = top_rt["class"] if top_rt else "Background"
        row["isnet_rt_conf"] = round(top_rt["score"], 4) if top_rt else 0.0
        row["isnet_rt_ms"] = round(float(runs[NAME_ISNET_RT]["latencies_ms"][idx]), 2)

    if HAS_GROUND_TRUTH:
        ev_v4 = metrics_by_scenario[NAME_V4]["op"]["per_image"][idx]
        ev_is = metrics_by_scenario[NAME_ISNET_SSD]["op"]["per_image"][idx]
        row["v4_iou_gt"] = ev_v4["top_iou"]
        row["isnet_ssd_iou_gt"] = ev_is["top_iou"]
        row["delta_iou"] = (
            round(ev_is["top_iou"] - ev_v4["top_iou"], 4)
            if (ev_v4["top_iou"] is not None and ev_is["top_iou"] is not None)
            else None
        )
        row["v4_correct"] = ev_v4["is_correct"]
        row["isnet_ssd_correct"] = ev_is["is_correct"]
        if RUN_RUNTIME_ISNET_SCENARIO:
            ev_rt = metrics_by_scenario[NAME_ISNET_RT]["op"]["per_image"][idx]
            row["isnet_rt_iou_gt"] = ev_rt["top_iou"]
            row["isnet_rt_correct"] = ev_rt["is_correct"]

        if ev_v4["is_correct"] and ev_is["is_correct"]:
            both_ok += 1
            row["status"] = "Keduanya Benar"
        elif ev_v4["is_correct"] and not ev_is["is_correct"]:
            v4_only_ok += 1
            row["status"] = "Hanya v4 Benar"
        elif not ev_v4["is_correct"] and ev_is["is_correct"]:
            isnet_only_ok += 1
            row["status"] = "Hanya IS-Net Benar"
        else:
            both_wrong += 1
            row["status"] = "Keduanya Salah"
    detail_rows.append(row)

detail_df = pd.DataFrame(detail_rows)

agreement_summary = {
    "class_agreement_pct_v4_vs_isnet_ssd": round(100.0 * class_agreements / max(1, len(test_samples)), 2),
    "mean_iou_between_v4_and_isnet_ssd": round(float(np.mean(inter_model_ious)), 4) if inter_model_ious else None,
}
if HAS_GROUND_TRUTH:
    mcnemar_p = exact_mcnemar_p(v4_only_ok, isnet_only_ok)
    agreement_summary.update({
        "both_correct": both_ok,
        "v4_only_correct": v4_only_ok,
        "isnet_ssd_only_correct": isnet_only_ok,
        "both_wrong": both_wrong,
        "mcnemar_p_value_v4_vs_isnet_ssd": round(mcnemar_p, 5),
        "statistically_significant_alpha_0_05": bool(mcnemar_p < 0.05),
    })

# Bangun Tabel Perbandingan Lintas Skenario
def _metric_row(label: str, vals_by_sc: dict, unit: str = "%", higher_better: bool = True, tol: float = 0.2):
    row = {"Metrik": label}
    for sc in ACTIVE_SCENARIOS:
        row[sc] = f"{vals_by_sc[sc]:.2f} {unit}".strip()
    delta_main = vals_by_sc[NAME_ISNET_SSD] - vals_by_sc[NAME_V4]
    sign = "+" if delta_main >= 0 else ""
    row["Δ (SSD IS-Net - v4)"] = f"{sign}{delta_main:.2f} {unit}".strip()
    best_val = max(vals_by_sc.values()) if higher_better else min(vals_by_sc.values())
    winners = [sc for sc, v in vals_by_sc.items() if abs(v - best_val) <= tol]
    row["Terbaik"] = "Setara" if len(winners) == len(ACTIVE_SCENARIOS) else " / ".join(winners)
    return row


comp_rows = []
if HAS_GROUND_TRUTH:
    comp_rows += [
        _metric_row("COCO mAP@[0.50:0.95]", {s: 100 * metrics_by_scenario[s]["coco"]["mAP"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("COCO AP@0.50", {s: 100 * metrics_by_scenario[s]["coco"]["AP50"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("COCO AP@0.75 (Lokalisasi Ketat)", {s: 100 * metrics_by_scenario[s]["coco"]["AP75"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("COCO Average Recall (AR@100)", {s: 100 * metrics_by_scenario[s]["coco"]["AR100"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("Akurasi End-to-End (conf>=0.50)", {s: 100 * metrics_by_scenario[s]["op"]["e2e_accuracy"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("Precision (conf>=0.50)", {s: 100 * metrics_by_scenario[s]["op"]["precision"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("Recall (conf>=0.50)", {s: 100 * metrics_by_scenario[s]["op"]["recall"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("F1-Score (conf>=0.50)", {s: 100 * metrics_by_scenario[s]["op"]["f1"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("F1-Score Optimal (Sweep)", {s: 100 * metrics_by_scenario[s]["best_f1_sweep"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("Mean IoU (True Positive)", {s: 100 * metrics_by_scenario[s]["op"]["mean_iou"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("Proporsi TP IoU >= 0.75", {s: 100 * metrics_by_scenario[s]["op"]["pct_iou_ge_75"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("Background False Positive Rate", {s: 100 * metrics_by_scenario[s]["op"]["bg_fpr"] for s in ACTIVE_SCENARIOS}, "%", False, 0.1),
    ]
    for c in CLASSES:
        comp_rows.append(
            _metric_row(f"AP@0.50 Kelas {c}", {s: 100 * metrics_by_scenario[s]["coco"]["per_class_AP50"][c] for s in ACTIVE_SCENARIOS}, "%", True, 0.2)
        )
else:
    comp_rows += [
        _metric_row("Detection Rate (conf>=0.50)", {s: 100 * metrics_by_scenario[s]["detection_rate"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
        _metric_row("Mean Top Confidence", {s: 100 * metrics_by_scenario[s]["mean_top_conf"] for s in ACTIVE_SCENARIOS}, "%", True, 0.2),
    ]

comp_rows += [
    _metric_row("Latency End-to-End Mean", {s: metrics_by_scenario[s]["mean_ms"] for s in ACTIVE_SCENARIOS}, "ms", False, 0.5),
    _metric_row("Latency End-to-End P95", {s: metrics_by_scenario[s]["p95_ms"] for s in ACTIVE_SCENARIOS}, "ms", False, 0.5),
    _metric_row("Kecepatan Real-Time", {s: metrics_by_scenario[s]["fps"] for s in ACTIVE_SCENARIOS}, "FPS", True, 0.5),
    _metric_row("Ukuran Total Deployment", {s: metrics_by_scenario[s]["total_size_mb"] for s in ACTIVE_SCENARIOS}, "MB", False, 0.5),
]

comparison_df = pd.DataFrame(comp_rows)
display(comparison_df)
print("\n=== ANALISIS KESEPAKATAN (AGREEMENT) & UJI STATISTIK McNEMAR ===")
print(json.dumps(agreement_summary, indent=2, ensure_ascii=False))

## Cell 6 — Grafik Perbandingan Akurasi Utama, Per Kelas, & Sensitivitas Threshold (300 DPI)

1. **`01_perbandingan_metrik_utama.png`**: Perbandingan seluruh metrik COCO & operasional antar skenario proyek.
2. **`02_per_kelas_dan_threshold_sweep.png`**: Panel kiri = `AP@0.50` per kelas (`Mobil`, `Truk`, `Bus`); Panel kanan = Kurva `F1-Score` terhadap `Confidence Threshold (0.25–0.80)`.

In [ ]:
if HAS_GROUND_TRUTH:
    # 1. Grafik Metrik Utama
    metric_keys = [
        ("mAP\n50:95", lambda m: 100 * m["coco"]["mAP"]),
        ("AP@0.50", lambda m: 100 * m["coco"]["AP50"]),
        ("AP@0.75", lambda m: 100 * m["coco"]["AP75"]),
        ("AR@100", lambda m: 100 * m["coco"]["AR100"]),
        ("Akurasi\nE2E", lambda m: 100 * m["op"]["e2e_accuracy"]),
        ("Precision", lambda m: 100 * m["op"]["precision"]),
        ("Recall", lambda m: 100 * m["op"]["recall"]),
        ("F1-Score", lambda m: 100 * m["op"]["f1"]),
        ("Mean\nIoU", lambda m: 100 * m["op"]["mean_iou"]),
        ("BG\nSpecif.", lambda m: 100 * m["op"]["bg_specificity"]),
    ]

    x = np.arange(len(metric_keys))
    n_sc = len(ACTIVE_SCENARIOS)
    w = 0.78 / n_sc

    fig, ax = plt.subplots(figsize=(16.0, 5.9))
    for i_sc, sc in enumerate(ACTIVE_SCENARIOS):
        vals = [fn(metrics_by_scenario[sc]) for _, fn in metric_keys]
        offset = (i_sc - (n_sc - 1) / 2.0) * w
        bars = ax.bar(
            x + offset, vals, width=w,
            color=SCENARIO_COLORS[sc], edgecolor="white",
            label=f"{sc} ({metrics_by_scenario[sc]['checkpoint']})",
        )
        for rect in bars:
            hv = rect.get_height()
            ax.text(
                rect.get_x() + rect.get_width() / 2, hv + 0.9,
                f"{hv:.1f}", ha="center", va="bottom",
                fontsize=7.6 if n_sc == 3 else 8.3, fontweight="bold", color="#0F172A",
            )

    ax.set_xticks(x)
    ax.set_xticklabels([k[0] for k in metric_keys], fontsize=9.5, fontweight="bold")
    ax.set_ylabel("Persentase (%)", fontsize=10.5, fontweight="bold")
    ax.set_ylim(0, 114)
    ax.set_title(
        f"Perbandingan Akurasi Utama: Proyek Saat Ini (v4) vs Proyek Ditambah IS-Net ({len(test_samples)} Gambar Uji)",
        fontsize=13, pad=12,
    )
    ax.legend(loc="lower right", frameon=True, facecolor="white", fontsize=9.2)
    plt.tight_layout()
    save_fig(fig, "01_perbandingan_metrik_utama.png")
    plt.show()

    # 2. Per Kelas AP@0.50 + Threshold Sweep F1
    fig, axes = plt.subplots(1, 2, figsize=(15.8, 5.4))

    xc = np.arange(len(CLASSES))
    wc = 0.74 / n_sc
    for i_sc, sc in enumerate(ACTIVE_SCENARIOS):
        vals_c = [100 * metrics_by_scenario[sc]["coco"]["per_class_AP50"][c] for c in CLASSES]
        offset = (i_sc - (n_sc - 1) / 2.0) * wc
        bars_c = axes[0].bar(xc + offset, vals_c, width=wc, color=SCENARIO_COLORS[sc], edgecolor="white", label=sc)
        for r in bars_c:
            hv = r.get_height()
            axes[0].text(r.get_x() + r.get_width() / 2, hv + 1.0, f"{hv:.1f}", ha="center", va="bottom", fontsize=8.3, fontweight="bold")
    axes[0].set_xticks(xc)
    axes[0].set_xticklabels(CLASSES, fontsize=10.5, fontweight="bold")
    axes[0].set_ylim(0, 112)
    axes[0].set_ylabel("AP@0.50 (%)", fontweight="bold")
    axes[0].set_title("Akurasi Per Kelas Kendaraan (COCO AP@0.50)", fontsize=11.5)
    axes[0].legend(loc="lower right", fontsize=8.8, frameon=True)

    markers = ["o", "s", "^"]
    for i_sc, sc in enumerate(ACTIVE_SCENARIOS):
        sub = sweep_df[sweep_df["Skenario"] == sc]
        col = SCENARIO_COLORS[sc]
        axes[1].plot(sub["Threshold"], sub["F1_pct"], marker=markers[i_sc % len(markers)], linewidth=2.3, color=col, label=f"{sc}")
        best_r = sub.loc[sub["F1_pct"].idxmax()]
        axes[1].scatter([best_r["Threshold"]], [best_r["F1_pct"]], s=85, zorder=5, edgecolor="black", facecolor=col)
        axes[1].annotate(
            f"{best_r['F1_pct']:.1f}% @{best_r['Threshold']:.2f}",
            xy=(best_r["Threshold"], best_r["F1_pct"]),
            xytext=(0, 8 + 9 * (i_sc % 2)), textcoords="offset points",
            ha="center", fontsize=8.0, fontweight="bold", color=col,
        )
    axes[1].axvline(CONF_THRESHOLD, color="#EF4444", linestyle=":", linewidth=1.8, label=f"Threshold Produksi ({CONF_THRESHOLD:.2f})")
    axes[1].set_xlabel("Confidence Threshold", fontweight="bold")
    axes[1].set_ylabel("F1-Score (%)", fontweight="bold")
    axes[1].set_ylim(max(0, sweep_df["F1_pct"].min() - 10), 108)
    axes[1].set_title("Sensitivitas F1-Score terhadap Confidence Threshold", fontsize=11.5)
    axes[1].legend(loc="lower left", fontsize=8.3, frameon=True)

    plt.tight_layout()
    save_fig(fig, "02_per_kelas_dan_threshold_sweep.png")
    plt.show()
else:
    print("Mode tanpa Ground Truth — melewati grafik COCO & threshold sweep.")

## Cell 7 — Confusion Matrix & Dashboard Dampak ke Proyek (Akurasi vs FPS Real-Time, Breakdown Latency, & Ukuran Model)

1. **`03_confusion_matrix_side_by_side.png`**: Membandingkan pola salah-kelas & false positive background di tiap skenario.
2. **`04_dampak_ke_proyek_akurasi_vs_fps_ukuran.png`**: Grafik kunci untuk keputusan arsitektur proyek FERQUEUE — memperlihatkan **trade-off Akurasi vs FPS**, **breakdown waktu (`IS-Net ms` vs `SSD ms`)**, serta **total ukuran deployment (`MB`)**.

In [ ]:
if HAS_GROUND_TRUTH:
    n_sc = len(ACTIVE_SCENARIOS)
    cmaps = ["Blues", "Greens", "Oranges"]
    fig, axes = plt.subplots(1, n_sc, figsize=(5.8 * n_sc, 5.1))
    if n_sc == 1:
        axes = [axes]
    for ax, sc, cmap in zip(axes, ACTIVE_SCENARIOS, cmaps):
        cm = metrics_by_scenario[sc]["op"]["cm"]
        cm_labels = metrics_by_scenario[sc]["op"]["cm_labels"]
        cm_pct = 100.0 * cm / np.maximum(1, cm.sum(axis=1, keepdims=True))

        ax.imshow(cm_pct, cmap=cmap, vmin=0, vmax=100)
        ax.set_xticks(range(len(cm_labels)))
        ax.set_yticks(range(len(cm_labels)))
        ax.set_xticklabels(cm_labels, rotation=20, ha="right", fontweight="bold")
        ax.set_yticklabels(cm_labels, fontweight="bold")
        ax.set_xlabel("Prediksi (conf >= 0.50, IoU >= 0.50)", fontweight="bold")
        ax.set_ylabel("Ground Truth", fontweight="bold")
        ax.set_title(
            f"{sc}\nAkurasi E2E = {100*metrics_by_scenario[sc]['op']['e2e_accuracy']:.1f}% | BG FPR = {100*metrics_by_scenario[sc]['op']['bg_fpr']:.1f}%",
            fontsize=10.2,
        )
        for r in range(len(cm_labels)):
            for c in range(len(cm_labels)):
                val, pct = cm[r, c], cm_pct[r, c]
                ax.text(c, r, f"{val}\n({pct:.0f}%)", ha="center", va="center", color=("white" if pct >= 55 else "#0F172A"), fontsize=8.4, fontweight="bold")
        ax.grid(False)

    fig.suptitle("Confusion Matrix Lintas Skenario Proyek FERQUEUE", fontsize=13, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.93])
    save_fig(fig, "03_confusion_matrix_side_by_side.png")
    plt.show()

# Dashboard Dampak ke Proyek: 3 Panel (Trade-off Akurasi vs FPS, Breakdown Latency, Total Ukuran Deployment)
fig, axes = plt.subplots(1, 3, figsize=(16.5, 5.1))

# Panel 1: Trade-off Akurasi vs FPS Real-Time
ax = axes[0]
for sc in ACTIVE_SCENARIOS:
    m = metrics_by_scenario[sc]
    y_acc = (100 * m["op"]["f1"]) if HAS_GROUND_TRUTH else (100 * m["detection_rate"])
    ax.scatter([m["fps"]], [y_acc], s=260, color=SCENARIO_COLORS[sc], edgecolor="#0F172A", linewidth=1.4, zorder=5, label=sc)
    ax.annotate(
        f"{sc}\n({m['fps']:.1f} FPS, {y_acc:.1f}%)",
        xy=(m["fps"], y_acc),
        xytext=(0, 12), textcoords="offset points", ha="center", fontsize=8.0, fontweight="bold",
        color="#0F172A",
    )
ax.axvline(REALTIME_TARGET_FPS, color="#EF4444", linestyle="--", linewidth=1.6, label=f"Target Real-Time ({REALTIME_TARGET_FPS:.0f} FPS)")
ax.set_xlabel(f"Kecepatan Inferensi End-to-End (FPS pada {DEVICE_LABEL})", fontweight="bold")
ax.set_ylabel("F1-Score (%)" if HAS_GROUND_TRUTH else "Detection Rate (%)", fontweight="bold")
ax.set_title("Trade-Off Akurasi vs Kecepatan Real-Time", fontsize=11.2)
ax.set_xlim(0, max(m["fps"] for m in metrics_by_scenario.values()) * 1.28 + 2)
ax.set_ylim(0, 114)
ax.legend(loc="lower right", fontsize=7.8, frameon=True)

# Panel 2: Stacked Bar Breakdown Waktu Inferensi (IS-Net ms + SSD ms)
ax = axes[1]
xs = np.arange(len(ACTIVE_SCENARIOS))
ssd_vals = [metrics_by_scenario[sc]["ssd_mean_ms"] for sc in ACTIVE_SCENARIOS]
isnet_vals = [metrics_by_scenario[sc]["isnet_mean_ms"] for sc in ACTIVE_SCENARIOS]
b_ssd = ax.bar(xs, ssd_vals, width=0.52, color="#3B82F6", label="Waktu SSD MobileNet V2 (ms)")
b_is = ax.bar(xs, isnet_vals, width=0.52, bottom=ssd_vals, color="#F59E0B", label="Tambahan Waktu IS-Net (ms)")
for i_sc in range(len(ACTIVE_SCENARIOS)):
    tot = ssd_vals[i_sc] + isnet_vals[i_sc]
    ax.text(xs[i_sc], tot + max(1.0, tot * 0.02), f"{tot:.1f} ms\n({1000/tot:.1f} FPS)", ha="center", va="bottom", fontsize=8.4, fontweight="bold")
ax.axhline(1000.0 / REALTIME_TARGET_FPS, color="#EF4444", linestyle="--", linewidth=1.4, label=f"Batas {1000/REALTIME_TARGET_FPS:.0f} ms")
ax.set_xticks(xs)
ax.set_xticklabels([sc.replace(" (", "\n(") for sc in ACTIVE_SCENARIOS], fontsize=8.5, fontweight="bold")
ax.set_ylabel("Waktu per Frame (ms) — Lebih Kecil Lebih Baik", fontweight="bold")
ax.set_title("Breakdown Latency End-to-End per Frame", fontsize=11.2)
ax.set_ylim(0, max(s + i for s, i in zip(ssd_vals, isnet_vals)) * 1.25 + 10)
ax.legend(loc="upper left", fontsize=8.0, frameon=True)

# Panel 3: Ukuran Total Model Deployment (MB)
ax = axes[2]
sizes = [metrics_by_scenario[sc]["total_size_mb"] for sc in ACTIVE_SCENARIOS]
cols_sz = [SCENARIO_COLORS[sc] for sc in ACTIVE_SCENARIOS]
bars_sz = ax.bar(xs, sizes, width=0.52, color=cols_sz, edgecolor="white")
for r in bars_sz:
    hv = r.get_height()
    ax.text(r.get_x() + r.get_width() / 2, hv + max(1.0, hv * 0.02), f"{hv:.1f} MB", ha="center", va="bottom", fontsize=8.8, fontweight="bold")
ax.set_xticks(xs)
ax.set_xticklabels([sc.replace(" (", "\n(") for sc in ACTIVE_SCENARIOS], fontsize=8.5, fontweight="bold")
ax.set_ylabel("Total Ukuran File Model (MB)", fontweight="bold")
ax.set_title("Beban Penyimpanan / Deployment Proyek", fontsize=11.2)
ax.set_ylim(0, max(sizes) * 1.22 + 5)

fig.suptitle(f"Dampak Nyata Penambahan IS-Net terhadap Proyek FERQUEUE ({DEVICE_LABEL})", fontsize=13.2, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.93])
save_fig(fig, "04_dampak_ke_proyek_akurasi_vs_fps_ukuran.png")
plt.show()

## Cell 8 — Perbandingan Visual Side-by-Side & Analisis Kasus Menang/Kalah

1. **`05_visual_side_by_side_representatif.png`**: Sampel lintas kelas (`Mobil`, `Truk`, `Bus`, `Background`) menampilkan **Ground Truth** vs **Proyek Saat Ini (`v4`)** vs **Proyek + SSD IS-Net (`NB7`)** (dan **IS-Net Runtime** bila aktif).
2. **`06_analisis_kasus_perbedaan.png`**: Kasus paling kontras — gambar saat **Proyek + SSD IS-Net (`NB7`) unggul** vs saat **Proyek Saat Ini (`v4`) unggul**.

In [ ]:
def draw_boxes_rgb(rgb: np.ndarray, boxes: list[dict], show_score: bool = True, override_color=None) -> np.ndarray:
    canvas = rgb.copy()
    h, w = canvas.shape[:2]
    thick = max(2, int(round(min(h, w) / 170)))
    font_scale = max(0.55, min(h, w) / 680.0)
    for b in boxes:
        x1, y1, x2, y2 = [int(round(v)) for v in b["xyxy_px"]]
        color = override_color if override_color is not None else CLASS_COLORS.get(b["class"], (255, 255, 255))
        cv2.rectangle(canvas, (x1, y1), (x2, y2), color, thick)
        lbl = f"{b['class']} {b['score']*100:.0f}%" if (show_score and "score" in b) else b["class"]
        (tw, th), _ = cv2.getTextSize(lbl, cv2.FONT_HERSHEY_SIMPLEX, font_scale, max(1, thick - 1))
        ly = max(th + 8, y1)
        cv2.rectangle(canvas, (x1, ly - th - 8), (x1 + tw + 10, ly), color, -1)
        cv2.putText(canvas, lbl, (x1 + 5, ly - 4), cv2.FONT_HERSHEY_SIMPLEX, font_scale, (15, 23, 42), max(1, thick - 1), cv2.LINE_AA)
    return canvas


def plot_scenario_grid(indices: list[int], row_notes: list[str], title: str, out_filename: str):
    if not indices:
        print(f"Tidak ada sampel untuk {out_filename}.")
        return
    n_rows = len(indices)
    n_cols = len(ACTIVE_SCENARIOS) + (1 if HAS_GROUND_TRUTH else 0)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.7 * n_cols, 4.0 * n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for r_i, (idx, note) in enumerate(zip(indices, row_notes)):
        s = test_samples[idx]
        row_info = detail_df.iloc[idx]
        c_pos = 0

        if HAS_GROUND_TRUTH:
            gt_vis = draw_boxes_rgb(s["rgb"], s["gt"], show_score=False, override_color=(245, 158, 11))
            gt_title = f"Ground Truth: {row_info['gt_class']}" + (f"\n[{note}]" if note else "")
            axes[r_i, c_pos].imshow(gt_vis)
            axes[r_i, c_pos].set_title(gt_title, fontsize=9.0)
            axes[r_i, c_pos].axis("off")
            c_pos += 1

        for sc in ACTIVE_SCENARIOS:
            dets = runs[sc]["prod"][idx]
            vis = draw_boxes_rgb(s["rgb"], dets, show_score=True)
            top = dets[0] if dets else None
            cls_s = top["class"] if top else "Background"
            conf_s = f" ({top['score']*100:.0f}%)" if top else ""
            ms_val = float(runs[sc]["latencies_ms"][idx])
            iou_s = ""
            if HAS_GROUND_TRUTH:
                ev_i = metrics_by_scenario[sc]["op"]["per_image"][idx]
                if ev_i["top_iou"] is not None:
                    iou_s = f" | IoU={ev_i['top_iou']:.2f}"
            axes[r_i, c_pos].imshow(vis)
            axes[r_i, c_pos].set_title(f"{sc}\n{cls_s}{conf_s}{iou_s} | {ms_val:.0f} ms", fontsize=8.8)
            axes[r_i, c_pos].axis("off")
            c_pos += 1

    fig.suptitle(title, fontsize=13.2, fontweight="bold")
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    save_fig(fig, out_filename)
    plt.show()


# 1. Sampel representatif lintas kelas
rep_indices, rep_notes = [], []
if HAS_GROUND_TRUTH:
    for target_cls in CLASSES + ["Background"]:
        sub = detail_df[detail_df["gt_class"] == target_cls]
        if not sub.empty:
            pref = sub[sub["status"] == "Keduanya Benar"] if "status" in sub.columns else sub
            pick = pref.iloc[0] if not pref.empty else sub.iloc[0]
            rep_indices.append(int(pick.name))
            rep_notes.append(f"Sampel {target_cls}")
else:
    for idx in range(min(4, len(test_samples))):
        rep_indices.append(idx)
        rep_notes.append(f"Sampel #{idx+1}")

plot_scenario_grid(
    rep_indices,
    rep_notes,
    "Perbandingan Visual Lintas Kelas: Proyek Saat Ini (v4) vs Proyek Ditambah IS-Net",
    "05_visual_side_by_side_representatif.png",
)

# 2. Analisis Kasus Menang / Kalah
if HAS_GROUND_TRUTH:
    diff_indices, diff_notes = [], []
    is_win = detail_df[detail_df["status"] == "Hanya IS-Net Benar"].head(2)
    for idx_row in is_win.index:
        diff_indices.append(int(idx_row))
        diff_notes.append("SSD IS-Net Unggul (Benar)")
    if len(is_win) < 2:
        rem = detail_df[detail_df["delta_iou"].notna() & (~detail_df.index.isin(diff_indices))].sort_values("delta_iou", ascending=False).head(2 - len(is_win))
        for idx_row, r in rem.iterrows():
            diff_indices.append(int(idx_row))
            diff_notes.append(f"SSD IS-Net Unggul IoU ({r['delta_iou']:+.2f})")

    v4_win = detail_df[detail_df["status"] == "Hanya v4 Benar"].head(2)
    for idx_row in v4_win.index:
        diff_indices.append(int(idx_row))
        diff_notes.append("Proyek v4 Unggul (Benar)")
    if len(v4_win) < 2:
        rem = detail_df[detail_df["delta_iou"].notna() & (~detail_df.index.isin(diff_indices))].sort_values("delta_iou", ascending=True).head(2 - len(v4_win))
        for idx_row, r in rem.iterrows():
            diff_indices.append(int(idx_row))
            diff_notes.append(f"Proyek v4 Unggul IoU ({r['delta_iou']:+.2f})")

    plot_scenario_grid(
        diff_indices,
        diff_notes,
        "Analisis Kasus Kontras: Keunggulan SSD IS-Net vs Proyek Saat Ini (v4)",
        "06_analisis_kasus_perbedaan.png",
    )

## Cell 9 — Matriks Keputusan Proyek FERQUEUE (300 DPI), Rekomendasi Otomatis, & Ekspor ZIP

Merender tabel keputusan lengkap (`07_matriks_keputusan_proyek.png`), mencetak rekomendasi arsitektur untuk proyekmu (apakah lebih baik **tetap seperti saat ini**, **pakai SSD hasil latih IS-Net**, atau **pasang IS-Net penuh di runtime**), dan mengekspor paket ZIP ke Google Drive (`MyDrive/ferqueue/perbandingan_isnet_vs_v4/`).

In [ ]:
tbl_show = comparison_df.copy()

fig_h = max(4.6, 0.42 * len(tbl_show) + 1.8)
fig_w = 15.6 if len(ACTIVE_SCENARIOS) == 3 else 14.5
fig, ax = plt.subplots(figsize=(fig_w, fig_h))
ax.axis("off")
ax.set_title(
    f"Matriks Keputusan Proyek FERQUEUE: Tetap Seperti Saat Ini (v4) vs Ditambah IS-Net\n"
    f"Dataset Uji: {len(test_samples)} Gambar | Perangkat: {DEVICE_LABEL} | Target Real-Time >= {REALTIME_TARGET_FPS:.0f} FPS",
    fontsize=12.2, fontweight="bold", pad=12,
)

table = ax.table(
    cellText=tbl_show.values,
    colLabels=tbl_show.columns,
    loc="center",
    cellLoc="center",
)
table.auto_set_font_size(False)
table.set_fontsize(8.6 if len(ACTIVE_SCENARIOS) == 3 else 9.0)
table.scale(1.0, 1.55)

last_col = len(tbl_show.columns) - 1
for (r_i, c_i), cell in table.get_celld().items():
    cell.set_edgecolor("#CBD5E1")
    if r_i == 0:
        cell.set_facecolor("#0C2744")
        cell.set_text_props(color="white", weight="bold")
    else:
        winner = str(tbl_show.iloc[r_i - 1]["Terbaik"])
        if c_i == last_col:
            if NAME_ISNET_SSD in winner and NAME_V4 not in winner:
                cell.set_facecolor("#DCFCE7")
                cell.set_text_props(color="#065F46", weight="bold")
            elif NAME_V4 in winner and NAME_ISNET_SSD not in winner:
                cell.set_facecolor("#DBEAFE")
                cell.set_text_props(color="#1E3A8A", weight="bold")
            elif NAME_ISNET_RT in winner and NAME_V4 not in winner:
                cell.set_facecolor("#FEF3C7")
                cell.set_text_props(color="#92400E", weight="bold")
            else:
                cell.set_facecolor("#F1F5F9")
                cell.set_text_props(color="#475569", weight="bold")
        else:
            cell.set_facecolor("#FFFFFF" if r_i % 2 == 1 else "#F8FAFC")
            cell.set_text_props(color="#0F172A", weight=("bold" if c_i == 0 else "normal"))

plt.tight_layout()
save_fig(fig, "07_matriks_keputusan_proyek.png")
plt.show()

# Kesimpulan & Rekomendasi Arsitektur Proyek
m_v4 = metrics_by_scenario[NAME_V4]
m_is = metrics_by_scenario[NAME_ISNET_SSD]
m_rt = metrics_by_scenario.get(NAME_ISNET_RT)

print("\n" + "=" * 78)
print("KESIMPULAN: PROYEK TETAP SEPERTI SAAT INI (v4) vs DITAMBAH IS-NET")
print("=" * 78)
if HAS_GROUND_TRUTH:
    d_map = 100.0 * (m_is["coco"]["mAP"] - m_v4["coco"]["mAP"])
    d_ap50 = 100.0 * (m_is["coco"]["AP50"] - m_v4["coco"]["AP50"])
    d_f1 = 100.0 * (m_is["op"]["f1"] - m_v4["op"]["f1"])
    d_iou = 100.0 * (m_is["op"]["mean_iou"] - m_v4["op"]["mean_iou"])
    d_fpr = 100.0 * (m_is["op"]["bg_fpr"] - m_v4["op"]["bg_fpr"])

    print(f"[1] Proyek Saat Ini (v4)      : mAP={100*m_v4['coco']['mAP']:.2f}% | AP@50={100*m_v4['coco']['AP50']:.2f}% | F1={100*m_v4['op']['f1']:.2f}% | IoU={100*m_v4['op']['mean_iou']:.2f}% | {m_v4['fps']:.1f} FPS | {m_v4['total_size_mb']:.1f} MB")
    print(f"[2] Proyek + SSD IS-Net (NB7) : mAP={100*m_is['coco']['mAP']:.2f}% ({d_map:+.2f}%) | AP@50={100*m_is['coco']['AP50']:.2f}% ({d_ap50:+.2f}%) | F1={100*m_is['op']['f1']:.2f}% ({d_f1:+.2f}%) | IoU={100*m_is['op']['mean_iou']:.2f}% ({d_iou:+.2f}%) | {m_is['fps']:.1f} FPS | {m_is['total_size_mb']:.1f} MB")
    if m_rt is not None:
        d_map_rt = 100.0 * (m_rt["coco"]["mAP"] - m_v4["coco"]["mAP"])
        d_f1_rt = 100.0 * (m_rt["op"]["f1"] - m_v4["op"]["f1"])
        print(
            f"[3] Proyek + IS-Net Runtime   : mAP={100*m_rt['coco']['mAP']:.2f}% ({d_map_rt:+.2f}%) | AP@50={100*m_rt['coco']['AP50']:.2f}% | "
            f"F1={100*m_rt['op']['f1']:.2f}% ({d_f1_rt:+.2f}%) | Latency={m_rt['mean_ms']:.1f} ms (IS-Net {m_rt['isnet_mean_ms']:.1f} + SSD {m_rt['ssd_mean_ms']:.1f}) | "
            f"{m_rt['fps']:.1f} FPS | {m_rt['total_size_mb']:.1f} MB"
        )
    print(
        f"    Uji McNemar (v4 vs SSD IS-Net): p = {agreement_summary['mcnemar_p_value_v4_vs_isnet_ssd']:.5f} "
        f"({'SIGNIFIKAN p<0.05' if agreement_summary['statistically_significant_alpha_0_05'] else 'tidak berbeda signifikan p>=0.05'}) | "
        f"BG FPR selisih {d_fpr:+.2f}%"
    )

    if d_map >= 0.5 and d_f1 >= 0.0:
        rekomendasi = (
            "REKOMENDASI PROYEK: Gunakan Skenario 2 (Proyek + Model SSD Hasil Latih IS-Net NB7) via pcv.py import-model. "
            "Model IS-Net mengungguli v4 tanpa menambah latency maupun ukuran deployment di kamera."
        )
    elif d_map >= -2.0 and d_f1 >= -1.5:
        rekomendasi = (
            "REKOMENDASI PROYEK: Model SSD hasil latih IS-Net (Skenario 2) SETARA dengan model v4 saat ini (selisih tipis), "
            "sehingga layak dijadikan pengganti maupun bukti bahwa auto-bbox IS-Net sangat efektif tanpa perlu menambah beban runtime IS-Net di kamera."
        )
    else:
        rekomendasi = (
            "REKOMENDASI PROYEK: Untuk produksi gate pelabuhan utama, PERTAHANKAN model saat ini (v4) karena mAP/F1 anotasi manual masih lebih tinggi; "
            "sedangkan IS-Net paling ideal diposisikan sebagai alat bantu auto-labeling/segmentasi offline (NB1B -> NB7), bukan beban wajib di runtime kamera."
        )
    if m_rt is not None and not m_rt["realtime_ok"]:
        rekomendasi += (
            f" Catatan: Menjalankan IS-Net di runtime kamera (Skenario 3) menurunkan kecepatan menjadi {m_rt['fps']:.1f} FPS "
            f"(di bawah target {REALTIME_TARGET_FPS:.0f} FPS) dan menambah ukuran model +{ISNET_ONNX_SIZE_MB:.0f} MB."
        )
else:
    rekomendasi = (
        f"REKOMENDASI (Tanpa GT): Kesepakatan kelas v4 vs SSD IS-Net = {agreement_summary['class_agreement_pct_v4_vs_isnet_ssd']:.1f}%. "
        "Sediakan test.tfrecord untuk evaluasi mAP & F1 penuh."
    )

print("\n" + rekomendasi)

# Ekspor CSV, JSON, & ZIP Lengkap
tbl_csv_path = OUT_DIR / "08_tabel_perbandingan_proyek.csv"
detail_csv_path = OUT_DIR / "08_detail_per_gambar.csv"
sweep_csv_path = OUT_DIR / "08_threshold_sweep.csv"
summary_json_path = OUT_DIR / "08_ringkasan_perbandingan.json"

tbl_show.to_csv(tbl_csv_path, index=False)
detail_df.to_csv(detail_csv_path, index=False)
if not sweep_df.empty:
    sweep_df.to_csv(sweep_csv_path, index=False)


def _clean_for_json(obj):
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, dict):
        return {k: _clean_for_json(v) for k, v in obj.items() if k not in ("tp_ious", "tp_confs", "fp_confs", "per_image")}
    if isinstance(obj, list):
        return [_clean_for_json(v) for v in obj]
    return obj


summary_payload = {
    "test_source": TEST_SOURCE_LABEL,
    "has_ground_truth": HAS_GROUND_TRUTH,
    "total_test_images": len(test_samples),
    "device": DEVICE_LABEL,
    "isnet_training_mode_detected": ISNET_TRAINING_MODE,
    "runtime_isnet_scenario_enabled": RUN_RUNTIME_ISNET_SCENARIO,
    "active_scenarios": ACTIVE_SCENARIOS,
    "production_postprocessing": {
        "confidence_threshold": CONF_THRESHOLD,
        "nms_iou_threshold": NMS_IOU_THRESHOLD,
        "max_detections": MAX_DETECTIONS,
        "class_agnostic_nms": CLASS_AGNOSTIC_NMS,
    },
    "scenarios": _clean_for_json(metrics_by_scenario),
    "agreement": agreement_summary,
    "isnet_training_audit": ISNET_AUDIT_DATA.get("isnet_vs_gt_iou_summary") if ISNET_AUDIT_DATA else None,
    "recommendation": rekomendasi,
}
summary_json_path.write_text(json.dumps(summary_payload, indent=2, ensure_ascii=False), encoding="utf-8")

zip_path = OUT_DIR.parent / "nb8_perbandingan_ssd_isnet_vs_v4.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.rglob("*")):
        if p.is_file():
            zf.write(p, p.relative_to(OUT_DIR).as_posix())

print(f"\nSeluruh grafik (300 DPI), CSV, & JSON dikemas di: {zip_path} ({zip_path.stat().st_size / (1024**2):.2f} MB)")

drive_out = Path("/content/drive/MyDrive/ferqueue/perbandingan_isnet_vs_v4")
if IN_COLAB and Path("/content/drive/MyDrive").is_dir():
    drive_out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_path, drive_out / zip_path.name)
    shutil.copy2(tbl_csv_path, drive_out / tbl_csv_path.name)
    shutil.copy2(summary_json_path, drive_out / summary_json_path.name)
    print(f"Salinan ZIP, CSV, & JSON juga disimpan ke Google Drive: {drive_out}")

# Opsional: hapus '#' di bawah jika ingin langsung unduh ZIP lewat browser Colab
# if IN_COLAB:
#     colab_files.download(str(zip_path))